# Limpieza de Datos

# 1. Transformación fechas a signos y eliminar filas
El objetivo es transformar las fechas de nacimiento a signos zodiacales, los datos que no posean la fecha de nacimiento generan el valor desconocido, el cual debe eliminarse al no ser utilizable en el dataset

In [1]:
import pandas as pd
import os

# Diccionario con las fechas de los signos zodiacales incliyendo Ofiuco 
# (https://www.stylist.co.uk/astrology/ophiuchus-13th-zodiac-sign/933266)
# Adaptado de las fechas 
signo_fechas = {
    "Aries": ((4, 19), (5, 13)),
    "Tauro": ((5, 14), (6, 21)),
    "Géminis": ((6, 22), (7, 20)),
    "Cáncer": ((7, 21), (8, 10)),
    "Leo": ((8, 11), (9, 16)),
    "Virgo": ((9, 17), (10, 31)),
    "Libra": ((11, 1), (11, 23)),
    "Escorpio": ((11, 24), (11, 29)),
    "Ofiuco": ((11, 30), (12, 17)),
    "Sagitario": ((12, 18), (1, 19)),
    "Capricornio": ((1, 20), (2, 16)),
    "Acuario": ((2, 17), (3, 11)),
    "Piscis": ((3, 12), (4, 18)),
}

dias_mes = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

def fecha_valida(dia, mes):
    return 1 <= mes <= 12 and 1 <= dia <= dias_mes[mes - 1]

def day_of_year(dia, mes):
    return sum(dias_mes[:mes - 1]) + dia

def obtener_signo(dia, mes):
    if not fecha_valida(dia, mes):
        return None  # o raise ValueError("Fecha inválida")

    fecha = day_of_year(dia, mes)

    for signo, ((mes_ini, dia_ini), (mes_fin, dia_fin)) in signo_fechas.items():
        inicio = day_of_year(dia_ini, mes_ini)
        fin = day_of_year(dia_fin, mes_fin)

        if inicio <= fin:
            if inicio <= fecha <= fin:
                return signo
        else:  # cruza el año
            if fecha >= inicio or fecha <= fin:
                return signo

    return None
try:
    # =============================================================================
    # Configuración de rutas
    # =============================================================================
    archivo_original = '../1_data_processed/v0_base_de_datos_unificada.csv'
    archivo_salida = '../1_data_processed/v1_ofiuco_base_filas_validas.csv'
    archivo_eliminadas = '../4_results/v1_ofiuco_filas_eliminadas.csv'

    # Asegurar que exista la carpeta de salida
    os.makedirs(os.path.dirname(archivo_salida), exist_ok=True)

    print("Cargando el archivo completo...")
    df = pd.read_csv(archivo_original, low_memory=False)
    total_filas_original = df.shape[0]
    total_cols_original = df.shape[1]

    # =============================================================================
    # 1. TRANSFORMACIÓN: FECHA -> SIGNO
    # =============================================================================
    print("--- Iniciando Transformación FECHA_NACIMIENTO -> SIGNO_ZODIACAL ---")

    if 'FECHA_NACIMIENTO' not in df.columns:
        raise KeyError("No existe la columna 'FECHA_NACIMIENTO' en el CSV original.")

    # Convertir FECHA_NACIMIENTO a datetime (inválidos -> NaT)
    df['FECHA_NACIMIENTO'] = pd.to_datetime(df['FECHA_NACIMIENTO'], errors='coerce')

    # Crear SIGNO_ZODIACAL
    df['SIGNO_ZODIACAL'] = df.apply(
        lambda row: obtener_signo(row['FECHA_NACIMIENTO'].day, row['FECHA_NACIMIENTO'].month)
        if pd.notna(row['FECHA_NACIMIENTO']) else 'Desconocido',
        axis=1
    )

    # =============================================================================
    # 2. LIMPIEZA DE FILAS: eliminar filas sin fecha válida o sin signo
    # =============================================================================
    print("--- Eliminando filas sin fecha válida o con signo 'Desconocido' ---")

    mask_invalidas = df['FECHA_NACIMIENTO'].isna() | (df['SIGNO_ZODIACAL'] == 'Desconocido')

    df_eliminadas = df.loc[mask_invalidas].copy()
    df_validas = df.loc[~mask_invalidas].copy()

    print(f"Filas originales: {total_filas_original:,}")
    print(f"Filas eliminadas: {df_eliminadas.shape[0]:,}")
    print(f"Filas válidas: {df_validas.shape[0]:,}")

    # Guardar eliminadas (opcional, pero recomendado)
    if df_eliminadas.shape[0] > 0:
        df_eliminadas.to_csv(archivo_eliminadas, index=False)
        print(f"✅ Filas eliminadas guardadas en: {archivo_eliminadas}")
        
    if "FECHA_NACIMIENTO" in df_validas.columns:
        df_validas.drop(columns=["FECHA_NACIMIENTO"], inplace=True)

    # Guardar dataset válido completo (NO se elimina ninguna columna)
    df_validas.to_csv(archivo_salida, index=False)

    print("\n" + "=" * 60)
    print("¡PROCESO COMPLETADO!")
    print(f"Archivo de salida: {archivo_salida}")
    print(f"Columnas mantenidas: {df_validas.shape[1]} (originales: {total_cols_original})")
    print("=" * 60)

except FileNotFoundError:
    print(f"Error: El archivo '{archivo_original}' no fue encontrado.")
except KeyError as e:
    print(f"Error: {str(e)}")
except Exception as e:
    print(f"Ocurrió un error inesperado: {str(e)}")

Cargando el archivo completo...
--- Iniciando Transformación FECHA_NACIMIENTO -> SIGNO_ZODIACAL ---
--- Eliminando filas sin fecha válida o con signo 'Desconocido' ---
Filas originales: 5,808,535
Filas eliminadas: 37
Filas válidas: 5,808,498
✅ Filas eliminadas guardadas en: ../4_results/v1_ofiuco_filas_eliminadas.csv

¡PROCESO COMPLETADO!
Archivo de salida: ../1_data_processed/v1_ofiuco_base_filas_validas.csv
Columnas mantenidas: 130 (originales: 130)


# Generar Archivo solo con las columnas necesarias

In [9]:
import pandas as pd
import os

archivo_csv = "../1_data_processed/v1_ofiuco_base_filas_validas.csv"
archivo_csv_slim = "../1_data_processed/v1_ofiuco_base_filas_validas_slim.csv"

os.makedirs("../1_data_processed/", exist_ok=True)

N_DIAG = 35
N_PROC = 30

diag_cols = [f"DIAGNOSTICO{i}" for i in range(1, N_DIAG + 1)]
proc_cols = [f"PROCEDIMIENTO{i}" for i in range(1, N_PROC + 1)]

cols_needed = ["SIGNO_ZODIACAL", "ESPECIALIDAD_MEDICA", "FECHA_INGRESO", "FECHAALTA"] + diag_cols + proc_cols
cols_needed_set = set(cols_needed)

chunksize = 250_000  # baja a 100_000 si aún pesa

# Si existe, lo sobreescribimos
if os.path.exists(archivo_csv_slim):
    os.remove(archivo_csv_slim)

total_rows = 0
first = True

for chunk in pd.read_csv(
    archivo_csv,
    usecols=lambda c: c in cols_needed_set,
    low_memory=False,
    chunksize=chunksize
):
    # Escribe por partes (append). No acumulamos en RAM.
    chunk.to_csv(archivo_csv_slim, mode="a", index=False, header=first)
    first = False

    total_rows += len(chunk)
    print(f"✅ Chunk escrito: {chunk.shape} | acumulado filas: {total_rows:,}")

print("✅ CSV slim guardado en:", archivo_csv_slim)

✅ Chunk escrito: (250000, 69) | acumulado filas: 250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 4,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 4,250,000
✅ C

# 2. Filtro de Columnas
Columnas a mantener asociadas a diagnósticos

In [10]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata

from sklearn.feature_selection import chi2
from sklearn.preprocessing import LabelEncoder

# =========================
# CONFIGURACIÓN
# =========================
archivo_entrada = "../1_data_processed/v1_ofiuco_base_filas_validas_slim.csv"

archivo_resultados_chi = "../4_results/v3_ofiuco_chi2_resultados.csv"
archivo_resumen_chi = "../4_results/v3_ofiuco_chi2_resumen_por_familia.csv"
archivo_dataset_final = "../1_data_processed/v3_ofiuco_dataset_post_chi.csv"

os.makedirs("../1_data_processed/", exist_ok=True)
os.makedirs("../4_results/Ofiuco/", exist_ok=True)

TARGET = "SIGNO_ZODIACAL"
ALPHA = 0.01  # umbral sugerido por tu profesor

SIGNOS_FIJOS = [
    "Acuario", "Aries", "Capricornio", "Cáncer",
    "Escorpio", "Géminis", "Leo", "Libra",
    "Piscis", "Sagitario", "Tauro", "Virgo", "Ofiuco"
]

N_DIAG = 35
N_PROC = 30

# =========================
# HELPERS
# =========================
def cie10_letra(codigo):
    """Extrae la primera letra A-Z del diagnóstico CIE."""
    if pd.isna(codigo):
        return "NA"
    s = str(codigo).strip().upper()
    if s == "" or s in {"NAN", "NONE"}:
        return "NA"
    m = re.search(r"[A-Z]", s)
    return m.group(0) if m else "OTROS"

def procedimiento_grupo(codigo):
    """
    Normaliza procedimiento a grupo 00–99 usando el bloque numérico inicial.

    Ejemplos:
      0.66  -> 00
      3.31  -> 03
      01.10 -> 01
      8.2   -> 08
      12    -> 12
    """
    if pd.isna(codigo):
        return "NA"

    s = str(codigo).strip()
    if s == "" or s in {"NAN", "NONE"}:
        return "NA"

    m = re.match(r"(\d+)", s)
    if not m:
        return "OTROS"

    try:
        num = int(m.group(1))
        if 0 <= num <= 99:
            return str(num).zfill(2)
        return "OTROS"
    except Exception:
        return "OTROS"

def normalizar_signo(s):
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_]", "", s)
    return s

def familia_variable(nombre):
    if nombre.startswith("ESPECIALIDAD_MEDICA_"):
        return "ESPECIALIDAD_MEDICA"
    elif nombre.startswith("DIAGNOSTICO"):
        return "DIAGNOSTICO"
    elif nombre.startswith("PROCEDIMIENTO"):
        return "PROCEDIMIENTO"
    else:
        return "OTRA"

# =========================
# PASO 0: CARGAR (solo columnas necesarias)
# =========================
diag_cols = [f"DIAGNOSTICO{i}" for i in range(1, N_DIAG + 1)]
proc_cols = [f"PROCEDIMIENTO{i}" for i in range(1, N_PROC + 1)]

cols_needed = [TARGET, "ESPECIALIDAD_MEDICA", "FECHA_INGRESO", "FECHAALTA"] + diag_cols + proc_cols
cols_needed_set = set(cols_needed)

df = pd.read_csv(
    archivo_entrada,
    usecols=lambda c: c in cols_needed_set,
    low_memory=False
)
print("Shape inicial:", df.shape)

missing_min = [c for c in [TARGET, "ESPECIALIDAD_MEDICA", "FECHA_INGRESO", "FECHAALTA"] if c not in df.columns]
if missing_min:
    raise KeyError(f"Faltan columnas mínimas en el CSV slim: {missing_min}")

# =========================
# PASO 1: ESTANCIA_DIAS
# =========================
df["FECHA_INGRESO"] = pd.to_datetime(df["FECHA_INGRESO"], errors="coerce")
df["FECHAALTA"] = pd.to_datetime(df["FECHAALTA"], errors="coerce")

df["ESTANCIA_DIAS"] = (df["FECHAALTA"] - df["FECHA_INGRESO"]).dt.days
df["ESTANCIA_DIAS"] = df["ESTANCIA_DIAS"].fillna(0).astype(np.int32)
df.loc[df["ESTANCIA_DIAS"] < 0, "ESTANCIA_DIAS"] = 0

# liberar fechas
df.drop(columns=["FECHA_INGRESO", "FECHAALTA"], inplace=True)

# =========================
# PASO 2: FILTRO BASE
# =========================
df = df[[TARGET, "ESPECIALIDAD_MEDICA", "ESTANCIA_DIAS"] + diag_cols + proc_cols].copy()
print("Shape después del filtro base:", df.shape)

# =========================
# PASO 3: TRANSFORMACIONES
# =========================
for c in diag_cols:
    df[c] = df[c].apply(cie10_letra)

for c in proc_cols:
    df[c] = df[c].apply(procedimiento_grupo)

print("✅ Transformación diagnósticos/procedimientos completada.")

# =========================
# PASO 4: ONE-HOT SPARSE
# =========================
cat_cols = ["ESPECIALIDAD_MEDICA"] + diag_cols + proc_cols

for c in cat_cols:
    df[c] = df[c].astype("category")

X_cat = pd.get_dummies(df[cat_cols], drop_first=False, sparse=True)
print("✅ One-hot (sparse) listo:", X_cat.shape)

# Convertir a CSR float32 antes de chi2
X_csr = X_cat.sparse.to_coo().tocsr().astype(np.float32)

# =========================
# PASO 5: CHI² SIMULTÁNEO
# =========================
le = LabelEncoder()
y_encoded = le.fit_transform(df[TARGET].astype("object"))

chi_scores, p_values = chi2(X_csr, y_encoded)

resultados = pd.DataFrame({
    "variable": X_cat.columns.astype(str),
    "chi2": chi_scores,
    "p_value": p_values
}).sort_values("chi2", ascending=False)

resultados["familia"] = resultados["variable"].apply(familia_variable)
resultados["significativa"] = resultados["p_value"] < ALPHA

resultados.to_csv(archivo_resultados_chi, index=False)
print("✅ Resultados chi guardados en:", archivo_resultados_chi)

# =========================
# PASO 6: RESUMEN PARA EVALUAR VARIABLES "SOBRE LA MARCHA"
# =========================
resumen = (
    resultados.groupby("familia")
    .agg(
        total_variables=("variable", "count"),
        significativas=("significativa", "sum")
    )
    .reset_index()
)

resumen["porcentaje_significativas"] = (
    resumen["significativas"] / resumen["total_variables"] * 100
).round(2)

resumen.to_csv(archivo_resumen_chi, index=False)
print("✅ Resumen por familia guardado en:", archivo_resumen_chi)
print("\nResumen por familia:")
print(resumen)

# =========================
# PASO 7: FILTRAR VARIABLES SIGNIFICATIVAS
# =========================
vars_significativas = resultados.loc[resultados["p_value"] < ALPHA, "variable"].tolist()
print(f"\n✅ Variables significativas con p < {ALPHA}: {len(vars_significativas)}")

X_post = X_cat[vars_significativas]

# =========================
# PASO 8: BINARIZAR TARGET (ONE-VS-REST)
# =========================
Y_bin = pd.DataFrame(index=df.index)

for signo in SIGNOS_FIJOS:
    colname = f"signo_zodiacal_{normalizar_signo(signo)}"
    Y_bin[colname] = (df[TARGET] == signo).astype(np.uint8)

print("✅ Targets binarios creados:", list(Y_bin.columns))

# =========================
# PASO 9: GENERAR DATASET FINAL DIRECTO
# =========================
df_final = pd.concat([X_post, df["ESTANCIA_DIAS"], Y_bin], axis=1)
df_final.to_csv(archivo_dataset_final, index=False)

print("✅ Dataset final POST_CHI guardado:", df_final.shape, "->", archivo_dataset_final)
print("✅ Proceso terminado") 

Shape inicial: (5808498, 69)
Shape después del filtro base: (5808498, 68)
✅ Transformación diagnósticos/procedimientos completada.
✅ One-hot (sparse) listo: (5808498, 3744)
✅ Resultados chi guardados en: ../4_results/v3_ofiuco_chi2_resultados.csv
✅ Resumen por familia guardado en: ../4_results/v3_ofiuco_chi2_resumen_por_familia.csv

Resumen por familia:
               familia  total_variables  significativas  \
0          DIAGNOSTICO              940             222   
1  ESPECIALIDAD_MEDICA              104              45   
2        PROCEDIMIENTO             2700             227   

   porcentaje_significativas  
0                      23.62  
1                      43.27  
2                       8.41  

✅ Variables significativas con p < 0.01: 494
✅ Targets binarios creados: ['signo_zodiacal_acuario', 'signo_zodiacal_aries', 'signo_zodiacal_capricornio', 'signo_zodiacal_cancer', 'signo_zodiacal_escorpio', 'signo_zodiacal_geminis', 'signo_zodiacal_leo', 'signo_zodiacal_libra', 'sig